# 🌌 Aurora · Image generation (SDXL / Flux, NSFW-capable)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subash1327/aurora/blob/main/notebooks/aurora_image.ipynb)

SDXL text-to-image and image-to-image with 🤗 diffusers, **no safety checker**, so adult (NSFW) content between adults is possible.
Pick a popular community SDXL checkpoint, or load any SDXL `.safetensors` from Hugging Face or Civitai, add LoRAs, batch, seed, and save to Drive.
An optional **Flux** section works on L4/A100.

**Runs on:** SDXL on a free **T4** (fp16, fp16-safe VAE). Flux on **L4/A100** (4-bit on ≤24 GB).

**Steps:** 1 → 2 → 2b (optional: upload LoRA/checkpoint files from your computer) → 3 (load model) → 4 (optional LoRAs) → 5 (txt2img) / 6a (upload an init image, optional) → 6 (img2img).

> ### ⚠️ Content policy (enforced)
> * **Sexual or suggestive content involving minors is prohibited.** Prompts, checkpoint names and LoRA names that contain minor/child-related terms (child, kid, teen, underage, loli, schoolgirl, ages under 18, …) are refused before generation. Rephrase if you hit a false positive.
> * **No sexual, nude or non-consensual imagery of real, identifiable people** (celebrities, acquaintances, anyone who has not consented), and no LoRAs trained on real people for that purpose.
> * You are responsible for complying with the model licenses and the law where you live.

In [ ]:
#@title 1. Install Aurora + dependencies (≈2 min)
AURORA_REF = "main"  #@param {type:"string"}
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!pip install -q -U -r https://raw.githubusercontent.com/subash1327/aurora/{AURORA_REF}/requirements.txt
!pip install -q -U --no-deps --force-reinstall "git+https://github.com/subash1327/aurora@{AURORA_REF}"
print("Installed. If Colab asks to restart the session, restart and continue from step 2.")

In [ ]:
#@title 2. Hugging Face login (Colab secret `HF_TOKEN`) + GPU check
#@markdown Add `HF_TOKEN` in the 🔑 **Secrets** sidebar and enable notebook access. Needed for gated models (e.g. FLUX).
from aurora.device import get_device_info, hf_login_from_colab_secret

hf_login_from_colab_secret("HF_TOKEN")
INFO = get_device_info()
print(INFO.summary())
if INFO.device != "cuda":
    print("⚠️  No GPU detected: Runtime > Change runtime type > T4 GPU (or L4 / A100).")
#@markdown For Civitai downloads add a `CIVITAI_TOKEN` secret too (Civitai > Account settings > API keys).

In [ ]:
#@title 2b. (Optional) Upload LoRA / checkpoint files from your computer
WHAT = "LoRA"  #@param ["LoRA", "checkpoint"]
UPLOAD_METHOD = "file picker"  #@param ["file picker", "drag & drop widget"]
#@markdown Then type `uploaded` (latest file) or `uploaded:2` (2nd file) in `CUSTOM_SOURCE` (step 3, checkpoints) or `LORA_1` / `LORA_2` (step 4).
#@markdown Files are saved under `/content/uploads/`. A 6 GB checkpoint uploads slowly through the browser; a Google Drive path (`/content/drive/MyDrive/...`) is faster.
from aurora.inputs import upload

if WHAT == "LoRA":
    LORA_UPLOADS = upload("lora", UPLOAD_METHOD, uploader=globals().get("LORA_UPLOADS"))
else:
    CKPT_UPLOADS = upload("checkpoint", UPLOAD_METHOD, uploader=globals().get("CKPT_UPLOADS"))

In [ ]:
#@title 3. Load a model
MODEL = "Juggernaut XL v9 (photoreal)"  #@param ["Juggernaut XL v9 (photoreal)", "RealVisXL V5.0 (photoreal)", "Animagine XL 4.0 (anime)", "Illustrious XL v0.1 (anime)", "DreamShaper XL v2 Turbo (fast)", "SDXL Base 1.0", "FLUX.1 schnell (L4/A100)", "FLUX.1 dev (L4/A100)", "Custom (CUSTOM_SOURCE below)"]
CUSTOM_SOURCE = ""  #@param {type:"string"}
#@markdown Custom: HF repo id (`owner/name`, diffusers format), HF file URL (`https://huggingface.co/.../blob/main/x.safetensors`), Civitai download URL (`https://civitai.com/api/download/models/<versionId>`), a local/Drive `.safetensors` path, or `uploaded` (file from step 2b).
CUSTOM_FAMILY = "sdxl"  #@param ["sdxl", "flux"]
SCHEDULER = "model default"  #@param ["model default", "DPM++ 2M Karras", "DPM++ 2M", "DPM++ SDE Karras", "Euler a", "Euler", "UniPC"]
DTYPE = "auto"  #@param ["auto", "fp16", "bf16", "fp32"]
OFFLOAD = "auto"  #@param ["auto", "none", "model", "sequential"]

from aurora.device import free_memory
from aurora.image import IMAGE_MODELS, ImageGenerator, set_scheduler
from aurora.inputs import resolve_source

_src = resolve_source(CUSTOM_SOURCE, globals().get("CKPT_UPLOADS"), "checkpoint") if MODEL.startswith("Custom") else MODEL
if not _src:
    raise ValueError("Set CUSTOM_SOURCE or pick a model from the list.")
_key = (_src, CUSTOM_FAMILY, DTYPE, OFFLOAD)
if globals().get("IMG") is None or globals().get("_IMG_KEY") != _key:
    if globals().get("IMG") is not None:
        IMG.unload()
        IMG = None
        free_memory()
    IMG = ImageGenerator(_src, family=CUSTOM_FAMILY, dtype=DTYPE, offload=OFFLOAD,
                         scheduler=None if SCHEDULER == "model default" else SCHEDULER)
    _IMG_KEY = _key
elif SCHEDULER != "model default" and IMG.family == "sdxl":
    set_scheduler(IMG.pipe, SCHEDULER)
    IMG._img2img = None
spec = IMAGE_MODELS.get(_src)
print(f"Ready: {_src} ({IMG.family}, {IMG.dtype}, offload={IMG.offload})" + (f" | {spec.notes} | default {spec.width}x{spec.height}, {spec.steps} steps, CFG {spec.guidance}" if spec else ""))

In [ ]:
#@title 4. (Optional) LoRAs
#@markdown HF repo id, HF file URL, Civitai download URL, local/Drive path, or `uploaded` / `uploaded:2` (files from step 2b). Leave blank to skip. Re-running replaces previously loaded LoRAs.
LORA_1 = ""  #@param {type:"string"}
LORA_1_WEIGHT = 0.8  #@param {type:"slider", min:-1.0, max:2.0, step:0.05}
LORA_1_FILE = ""  #@param {type:"string"}
#@markdown `LORA_x_FILE`: file name inside an HF repo that has several `.safetensors` (optional).
LORA_2 = ""  #@param {type:"string"}
LORA_2_WEIGHT = 0.8  #@param {type:"slider", min:-1.0, max:2.0, step:0.05}
LORA_2_FILE = ""  #@param {type:"string"}

from aurora.inputs import resolve_source

IMG.clear_loras()
for src, wt, fn in ((LORA_1, LORA_1_WEIGHT, LORA_1_FILE), (LORA_2, LORA_2_WEIGHT, LORA_2_FILE)):
    src = resolve_source(src, globals().get("LORA_UPLOADS"), "lora")
    if src:
        IMG.add_lora(src, weight=wt, weight_name=fn.strip() or None)
IMG._img2img = None
print("Active LoRAs:", IMG.loras or "none")

In [ ]:
#@title 5. Text → image
PROMPT = "photo of an elegant woman in a red evening dress on a rooftop at night, city lights bokeh, 85mm, film grain"  #@param {type:"string"}
NEGATIVE_PROMPT = "default"  #@param {type:"string"}
#@markdown `default` = generic quality negatives. Not used by Flux.
ADULT_SAFETY_NEGATIVES = True  #@param {type:"boolean"}
#@markdown Appends `child, teen, underage, …` to the negative prompt (recommended for NSFW work).
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
#@markdown `0` = model default (SDXL likes ~1 MP: 1024x1024, 832x1216, 1216x832, 896x1152).
STEPS = 0  #@param {type:"integer"}
GUIDANCE = -1  #@param {type:"number"}
#@markdown `0` steps / `-1` guidance = model default.
SEED = -1  #@param {type:"integer"}
BATCH = 2  #@param {type:"slider", min:1, max:8, step:1}
#@markdown T4: keep batch ≤ 4 at 1024². Seeds are SEED, SEED+1, …
CLIP_SKIP = 0  #@param {type:"integer"}
#@markdown A1111-style clip skip (2 for many anime models); 0 = model default.
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

import time

from aurora.image import DEFAULT_SDXL_NEGATIVE
from aurora.outputs import copy_to_drive, save_image, show_images
from aurora.safety import ADULT_NEGATIVE_PROMPT

neg = DEFAULT_SDXL_NEGATIVE if NEGATIVE_PROMPT.strip() == "default" else NEGATIVE_PROMPT
if ADULT_SAFETY_NEGATIVES:
    neg = f"{neg}, {ADULT_NEGATIVE_PROMPT}" if neg else ADULT_NEGATIVE_PROMPT

t0 = time.time()
images, metas = IMG.generate(PROMPT, negative_prompt=neg, width=WIDTH or None, height=HEIGHT or None,
                             steps=STEPS or None, guidance=None if GUIDANCE < 0 else GUIDANCE, seed=SEED,
                             batch_size=BATCH, clip_skip=CLIP_SKIP or None)
paths = [save_image(im, PROMPT, seed=m["seed"], metadata=m) for im, m in zip(images, metas)]
print(f"{len(images)} image(s) in {time.time() - t0:.0f}s | seeds {[m['seed'] for m in metas]}")
for p in paths:
    print("Saved:", p)
    if SAVE_TO_DRIVE:
        copy_to_drive(p, "Aurora/images")
show_images(images)

In [ ]:
#@title 6a. (Optional) Upload an init image from your computer
UPLOAD_METHOD = "file picker"  #@param ["file picker", "drag & drop widget"]
#@markdown `file picker` opens your computer's file dialog. `drag & drop widget` shows a drop zone; drop the file, then run step 6.
CLEAR_PREVIOUS = True  #@param {type:"boolean"}
#@markdown An uploaded image takes precedence over `INIT_IMAGE` in step 6 (untick `USE_UPLOADED` there to ignore it).
from aurora.inputs import upload

if CLEAR_PREVIOUS and globals().get("IMAGE_UPLOADS") is not None:
    IMAGE_UPLOADS.clear()
IMAGE_UPLOADS = upload("image", UPLOAD_METHOD, uploader=globals().get("IMAGE_UPLOADS"))

In [ ]:
#@title 6. Image → image
INIT_IMAGE = ""  #@param {type:"string"}
#@markdown URL, `/content/...` path, Drive path (`/content/drive/MyDrive/...`) or `last` (first image from step 5). An image uploaded in step 6a wins; if nothing is set, a file picker opens.
USE_UPLOADED = True  #@param {type:"boolean"}
PROMPT = "same woman, oil painting style, dramatic lighting"  #@param {type:"string"}
NEGATIVE_PROMPT = "default"  #@param {type:"string"}
ADULT_SAFETY_NEGATIVES = True  #@param {type:"boolean"}
STRENGTH = 0.55  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
#@markdown Lower = closer to the input image.
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
STEPS = 0  #@param {type:"integer"}
GUIDANCE = -1  #@param {type:"number"}
SEED = -1  #@param {type:"integer"}
BATCH = 1  #@param {type:"slider", min:1, max:8, step:1}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

from PIL import Image

from aurora.image import DEFAULT_SDXL_NEGATIVE
from aurora.outputs import copy_to_drive, save_image, show_images
from aurora.safety import ADULT_NEGATIVE_PROMPT

from aurora.inputs import resolve_image

init = resolve_image(INIT_IMAGE, uploader=globals().get("IMAGE_UPLOADS") if USE_UPLOADED else None,
                     last=globals().get("images", [None])[0])
if init is None:
    raise ValueError("img2img needs an init image: upload one in step 6a or set INIT_IMAGE.")
if isinstance(init, Image.Image) and not (WIDTH and HEIGHT):
    # keep the input's aspect ratio at ~1 MP
    r = (1024 * 1024 / (init.width * init.height)) ** 0.5
    WIDTH, HEIGHT = int(init.width * r) // 64 * 64, int(init.height * r) // 64 * 64

neg = DEFAULT_SDXL_NEGATIVE if NEGATIVE_PROMPT.strip() == "default" else NEGATIVE_PROMPT
if ADULT_SAFETY_NEGATIVES:
    neg = f"{neg}, {ADULT_NEGATIVE_PROMPT}" if neg else ADULT_NEGATIVE_PROMPT
out, metas = IMG.generate(PROMPT, negative_prompt=neg, init_image=init, strength=STRENGTH,
                          width=WIDTH or None, height=HEIGHT or None, steps=STEPS or None,
                          guidance=None if GUIDANCE < 0 else GUIDANCE, seed=SEED, batch_size=BATCH)
for im, m in zip(out, metas):
    p = save_image(im, PROMPT, seed=m["seed"], metadata=m)
    print("Saved:", p)
    if SAVE_TO_DRIVE:
        copy_to_drive(p, "Aurora/images")
show_images(out)

## Optional: Flux

Pick **FLUX.1 schnell** or **FLUX.1 dev** in step 3 (or `CUSTOM_FAMILY = "flux"` with a Flux transformer `.safetensors`).

* Needs an **L4 or A100** runtime (bf16). On ≤ 24 GB GPUs Aurora loads the transformer and T5 encoder in 4-bit NF4 automatically.
* Both repos are **gated**: accept the license on Hugging Face and add `HF_TOKEN` (step 2). FLUX.1-dev is non-commercial.
* schnell: 4 steps, guidance 0. dev: ~28 steps, guidance 3-4. Negative prompts are ignored.
* Flux LoRAs load the same way as SDXL LoRAs in step 4 (they must be Flux LoRAs).

## Troubleshooting

* **CUDA out of memory**: lower `BATCH`, use ~1 MP sizes, set `OFFLOAD = "model"` (or `"sequential"`) and re-run step 3. *Runtime > Restart session* frees fragmented VRAM.
* **Black images on T4**: fp16 overflow. Aurora already swaps in the fp16-safe SDXL VAE; if a custom checkpoint still produces black images set `DTYPE = "fp32"` + `OFFLOAD = "model"`.
* **Civitai 401/403**: add a `CIVITAI_TOKEN` Colab secret; use the *download* URL (`/api/download/models/<versionId>`), not the page URL.
* **Prompt refused**: the minor-safety filter matched a term (it is deliberately strict, e.g. `teen`, `kid`, `schoolgirl`, ages < 18). Rephrase; there is no override.
* **Long prompts**: SDXL's CLIP encoders read the first 77 tokens; put the important words first.
